# Guvenlik ve Prioritizasyon (Analiz 8)

**Tarih:** 2026-05-11
**Amac:** Filodaki en riskli araclari kompozit kritiklik skoru ile sıralamak.
**Cikti:** Top 50/100/tum lista + ML V6 icin `kritiklik_skoru` feature.

## Plan
- **KISIM A** (Bolum 1-4): Tek boyutlu risk skorlari hesabi
- **KISIM B** (Bolum 5-8): Kompozit skor 3 farklı agirlik politikasiyla
- **KISIM C** (Bolum 9-12): Operasyonel ciktilar (top liste, garaj, hat, aksiyon)
- **KISIM D** (Bolum 13-14): ML V6 feature + kisitlamalar

## Hipotezler (Onceden Belirlendi)
- H1: Kompozit skor yas tek basına'dan AUC olarak daha iyi
- H2: Top 50 risk arac ortalama ciddi_oran > %60 (genel %38 × 1.5x)
- H3: Risk Sahinkaya + Anadolu'da yogunlasiyor (Analiz 5/7 ile tutarli)
- H4: Kompozit skor leakage'lı değil (zaman-aware dususu <%30)

## Birlestirilen Feature'lar (Onceki Analizlerden)
- `yas` (temel)
- `gecmis_ciddi_oran` (zaman-aware: 30/60/90 gun pencereleri)
- `egim_maruziyet` (Analiz 3)
- `son_kaza_gun` (Analiz 4)
- `garaj_sistem_lift` (Analiz 5)
- `verimsizlik_skoru` (Analiz 7)

## Veri Notu
- ARACTIPI ayrımı yapılmıyor (proje kararı)
- ADALAR yok (zaten ariza_model.csv'de degil)
- yolcu_gunluk_doluluk.csv güvenilmez (proje kararı)


---
## 1. Veri Yukleme + Tum Feature'lari Birlestirme

Onceki analizlerden cikan kanitlanmis feature'lari toplayıp her arac icin tek satıra getiriyoruz.


In [1]:
# BOLUM 1: Veri yukleme + feature birlestirme
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import json as _json
import statsmodels.api as sm
from statsmodels.formula.api import ols

# Ana veri
df = pd.read_csv('../panel_data/temiz_veri/ariza_model.csv', low_memory=False)
df['OLAYTARIHI'] = pd.to_datetime(df['OLAYTARIHI'], format='mixed')
# Bilinmiyor YAKITTURU duzeltmesi (Analiz 7'den)
mask_b = df['YAKITTURU']=='Bilinmiyor'
df.loc[mask_b & df['MODEL'].str.contains('CNG', na=False), 'YAKITTURU'] = 'CNG'
df.loc[mask_b & ~df['MODEL'].str.contains('CNG', na=False), 'YAKITTURU'] = 'MOTORIN'
df = df[df['YAKITTURU'].isin(['MOTORIN','CNG'])].copy()
print(f'Ariza: {len(df):,}, arac: {df["KAPINO"].nunique():,}')

# Arac bazli temel profil
arac = df.groupby('KAPINO').agg(
    MARKA=('MARKA','first'),
    MODEL=('MODEL','first'),
    MODELYILI=('MODELYILI','first'),
    ARACCINSI=('ARACCINSI','first'),
    GARAJ=('GARAJ','first'),
    YAKITTURU=('YAKITTURU','first'),
    n_ariza=('ciddi_ariza','count'),
    ort_skor=('ciddiyet_skoru','mean'),
    ciddi_oran=('ciddi_ariza','mean'),
).reset_index()
arac['yas'] = 2025 - arac['MODELYILI']
print(f'Arac profili: {len(arac):,}')
print(arac.head())


Ariza: 58,557, arac: 3,508
Arac profili: 3,508
  KAPINO MARKA MODEL  MODELYILI ARACCINSI       GARAJ YAKITTURU  n_ariza  \
0  A3400  AKIA  LF25     2022.0   KORUKLU  Edirnekapı   MOTORIN       22   
1  A3401  AKIA  LF25     2022.0   KORUKLU  Edirnekapı   MOTORIN       11   
2  A3402  AKIA  LF25     2022.0   KORUKLU  Edirnekapı   MOTORIN        9   
3  A3403  AKIA  LF25     2022.0   KORUKLU  Edirnekapı   MOTORIN       23   
4  A3404  AKIA  LF25     2022.0   KORUKLU  Edirnekapı   MOTORIN        3   

   ort_skor  ciddi_oran  yas  
0  3.434091    0.363636  3.0  
1  4.155455    0.545455  3.0  
2  4.238889    0.555556  3.0  
3  2.993913    0.304348  3.0  
4  3.716667    0.333333  3.0  


---
## 2. Onceki Analizlerden Feature Entegrasyonu

Analiz 3 (egim_maruziyet) + Analiz 7 (verimsizlik_skoru + tuketim) + Analiz 5 garaj feature.


In [2]:
# BOLUM 2: Onceki feature'lari hesapla
# A) egim_maruziyet (Analiz 3)
with open(r'panel_data\hat_elevation.json', encoding='utf-8') as f:
    he_raw = _json.load(f)
he = pd.DataFrame([
    {'HATKODU': k, 'rakim': v.get('rakım_farkı', 0), 'tirm': v.get('tırmanma_m', 0)}
    for k, v in he_raw.items()
])
def mm_norm(s, q=None):
    if q is not None: s = s.clip(upper=s.quantile(q))
    mn, mx = s.min(), s.max()
    return ((s - mn) / (mx - mn) * 100).round(1) if mx > mn else pd.Series(0.0, index=s.index)
he['norm_r'] = mm_norm(he['rakim'], q=0.99)
he['norm_t'] = mm_norm(he['tirm'], q=0.99)
he['egim_puan'] = (he['norm_r']*0.4 + he['norm_t']*0.6).round(1)

ah = pd.read_csv('../panel_data/temiz_veri/arac_gunluk_hatlar.csv', low_memory=False)
ah = ah.merge(he[['HATKODU','egim_puan']], on='HATKODU', how='left')
arac_egim = ah.dropna(subset=['egim_puan']).groupby('KAPINO').apply(
    lambda g: np.average(g['egim_puan'], weights=g['SEFER_SAYISI'].clip(lower=0.01))
).reset_index(name='egim_maruziyet')
print(f'egim_maruziyet: {len(arac_egim):,} arac')

# B) verimsizlik_skoru (Analiz 7)
tuketim = pd.DataFrame([
    ('OTOKAR','KENT 290LF',40),('OTOKAR','KENT XL',60),
    ('MERCEDES','CITARO 0530',39),('MERCEDES','CITARO 0530 G',58),
    ('MERCEDES','CONECTO G',62),('MERCEDES','CONECTO',42),
    ('MERCEDES','CAPACITY',65),
    ('BMC','PROCITY TR',41),('BMC','PROCITY',41),
    ('KARSAN','AVANCITY S PLUS',58),('KARSAN','AVANCITY CNG',52),
    ('TEMSA','AVENUE LF CNG',50),
    ('AKIA','ULTRA LF12',40),('AKIA','LF25',60),
], columns=['MARKA','MODEL','tuketim_100km'])
arac = arac.merge(tuketim, on=['MARKA','MODEL'], how='left')
arac['tuketim_100km'] = arac['tuketim_100km'].fillna(arac['tuketim_100km'].median())
def norm(s):
    return ((s - s.min()) / (s.max() - s.min()) * 100).round(1) if s.max() > s.min() else pd.Series(0.0, index=s.index)
arac['yas_norm'] = norm(arac['yas'])
arac['tuketim_norm'] = norm(arac['tuketim_100km'])
arac['verimsizlik_skoru'] = (arac['yas_norm'] * arac['tuketim_norm'] / 100).round(2)

# C) garaj_sistem_lift (Analiz 5'ten kisaltilmis) - sistem yerine genel garaj_ort_skor
garaj_ort = df.groupby('GARAJ')['ciddiyet_skoru'].mean().reset_index()
garaj_ort.columns = ['GARAJ','garaj_ort_skor']
arac = arac.merge(garaj_ort, on='GARAJ', how='left')
arac = arac.merge(arac_egim, on='KAPINO', how='left')
arac['egim_maruziyet'] = arac['egim_maruziyet'].fillna(arac['egim_maruziyet'].median())
print(f'Birlestirilmis arac profili: {len(arac):,}, kolonlar: {list(arac.columns)}')
print(arac.describe().round(2))


egim_maruziyet: 6,760 arac
Birlestirilmis arac profili: 3,508, kolonlar: ['KAPINO', 'MARKA', 'MODEL', 'MODELYILI', 'ARACCINSI', 'GARAJ', 'YAKITTURU', 'n_ariza', 'ort_skor', 'ciddi_oran', 'yas', 'tuketim_100km', 'yas_norm', 'tuketim_norm', 'verimsizlik_skoru', 'garaj_ort_skor', 'egim_maruziyet']
       MODELYILI  n_ariza  ort_skor  ciddi_oran      yas  tuketim_100km  \
count    3508.00  3508.00   3508.00     3508.00  3508.00        3508.00   
mean     2013.44    16.69      3.56        0.36    11.56          48.80   
std         4.61     9.25      0.75        0.17     4.61           9.92   
min      2006.00     1.00      1.00        0.00     1.00          39.00   
25%      2012.00     9.00      3.16        0.25    10.00          40.00   
50%      2013.00    15.00      3.59        0.37    12.00          41.00   
75%      2015.00    23.00      3.99        0.47    13.00          60.00   
max      2024.00    58.00      8.45        1.00    19.00          65.00   

       yas_norm  tuketim_nor

C:\Users\asus\AppData\Local\Temp\ipykernel_7200\3058583610.py:19: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  arac_egim = ah.dropna(subset=['egim_puan']).groupby('KAPINO').apply(


---
## 3. Zaman-Aware Gecmis Ciddi Arıza Oranı (30/60/90 Gun)

Veri 6 ay -> Train (ilk 3 ay) icindeki performansa bakacagiz, sonra leakage kontrolu icin Test (son 3 ay) ile dogrulayacagiz.

Her arac icin **train doneminde** ciddi_oran (30, 60, 90 gunluk rolling).


In [3]:
# BOLUM 3: Zaman-aware gecmis ciddi_oran
df_sort = df.sort_values('OLAYTARIHI').reset_index(drop=True)
median_tarih = df_sort['OLAYTARIHI'].median()
print(f'Train/Test split tarihi (median): {median_tarih}')
train_df = df_sort[df_sort['OLAYTARIHI'] < median_tarih].copy()
test_df = df_sort[df_sort['OLAYTARIHI'] >= median_tarih].copy()
print(f'Train: {len(train_df):,} ariza, {train_df["KAPINO"].nunique()} arac')
print(f'Test:  {len(test_df):,} ariza, {test_df["KAPINO"].nunique()} arac')

# Train doneminde gecmis_ciddi_oran (full train donemine bak)
train_arac = train_df.groupby('KAPINO').agg(
    gecmis_n_ariza=('ciddi_ariza','count'),
    gecmis_ciddi_oran=('ciddi_ariza','mean'),
    gecmis_ort_skor=('ciddiyet_skoru','mean'),
).reset_index()
print(f'\nTrain arac feature: {len(train_arac):,}')
print(train_arac.describe().round(2))

# Test doneminde hedef (validation icin)
test_arac = test_df.groupby('KAPINO').agg(
    test_n_ariza=('ciddi_ariza','count'),
    test_ciddi_oran=('ciddi_ariza','mean'),
    test_ort_skor=('ciddiyet_skoru','mean'),
).reset_index()
print(f'\nTest arac feature: {len(test_arac):,}')

# Arac master birlesimi
arac = arac.merge(train_arac, on='KAPINO', how='left')
arac = arac.merge(test_arac, on='KAPINO', how='left')
# Eksikleri 0 ile doldur (o donemde arıza yapmamis)
for c in ['gecmis_n_ariza','gecmis_ciddi_oran','gecmis_ort_skor','test_n_ariza','test_ciddi_oran','test_ort_skor']:
    arac[c] = arac[c].fillna(0)
print(f'\nArac birlestirilmis: {len(arac):,}')


Train/Test split tarihi (median): 2025-04-07 17:16:17.798000128
Train: 29,279 ariza, 3456 arac
Test:  29,278 ariza, 3439 arac

Train arac feature: 3,456
       gecmis_n_ariza  gecmis_ciddi_oran  gecmis_ort_skor
count         3456.00            3456.00          3456.00
mean             8.47               0.36             3.56
std              5.19               0.22             0.99
min              1.00               0.00             1.00
25%              5.00               0.20             3.00
50%              8.00               0.35             3.54
75%             11.00               0.50             4.11
max             31.00               1.00             8.45

Test arac feature: 3,439

Arac birlestirilmis: 3,508


---
## 4. Tek Boyutlu Risk Skorları - Bireysel Korelasyonlar

Her feature'in test_ort_skor ile korelasyonunu olcuyoruz. Bu bize agirlik politikasi B icin r degerlerini verir.


In [4]:
# BOLUM 4: Bireysel feature × test_ort_skor korelasyonu
features = ['yas','gecmis_ciddi_oran','gecmis_ort_skor','egim_maruziyet','garaj_ort_skor','verimsizlik_skoru']
print('=== FEATURE x TEST_ORT_SKOR KORELASYON (zaman-aware) ===')
print(f'{"Feature":25s} {"Pearson_r":>10s} {"p":>10s} {"|r|":>8s}')
print('-'*60)

r_dict = {}
for f in features:
    s = arac[f].fillna(0)
    y = arac['test_ort_skor'].fillna(0)
    r, p = stats.pearsonr(s, y)
    r_dict[f] = abs(r)
    sig = '*' if p < 0.05 else ' '
    print(f'{f:25s} {r:+10.4f} {p:>10.4f} {abs(r):>8.4f} {sig}')

# Baseline AUC: sadece yas
from sklearn.metrics import roc_auc_score
# test_ciddi_oran > 0 -> riskli
arac['test_risk_binary'] = (arac['test_ciddi_oran'] >= arac['test_ciddi_oran'].median()).astype(int)
print(f'\nTest risk binary (median split): {arac["test_risk_binary"].sum()}/{len(arac)}')

valid = arac['test_n_ariza'] > 0
print(f'\nValid (test\'te ariza yapan) arac: {valid.sum()}')
y_true = arac.loc[valid, 'test_risk_binary']
for f in features:
    x = arac.loc[valid, f].fillna(0)
    if x.nunique() > 1:
        auc = roc_auc_score(y_true, x)
        print(f'AUC ({f:25s} → test_risk): {auc:.4f}')


=== FEATURE x TEST_ORT_SKOR KORELASYON (zaman-aware) ===
Feature                    Pearson_r          p      |r|
------------------------------------------------------------
yas                          +0.1692     0.0000   0.1692 *
gecmis_ciddi_oran            +0.1233     0.0000   0.1233 *
gecmis_ort_skor              +0.1596     0.0000   0.1596 *
egim_maruziyet               +0.1048     0.0000   0.1048 *
garaj_ort_skor               +0.3752     0.0000   0.3752 *
verimsizlik_skoru            +0.1279     0.0000   0.1279 *

Test risk binary (median split): 1769/3508

Valid (test'te ariza yapan) arac: 3439
AUC (yas                       → test_risk): 0.5288
AUC (gecmis_ciddi_oran         → test_risk): 0.5653
AUC (gecmis_ort_skor           → test_risk): 0.5513
AUC (egim_maruziyet            → test_risk): 0.5769
AUC (garaj_ort_skor            → test_risk): 0.5607
AUC (verimsizlik_skoru         → test_risk): 0.6003


---
## 5. KISIM B: Kompozit Skor - 3 Farklı Agirlik Politikasi

A. **Esit:** Her feature 1/N (basit baseline)
B. **r-Orantılı:** Feature'in test_ort_skor ile mutlak korelasyonuna oran
C. **Logistic Regression:** Test risk binary'sini ogrenip katsayilari agirlik olarak kullan


In [5]:
# BOLUM 5: 3 Kompozit Skor
features = ['yas','gecmis_ciddi_oran','gecmis_ort_skor','egim_maruziyet','garaj_ort_skor','verimsizlik_skoru']

# Normalize ALL features (0-100)
X = pd.DataFrame()
for f in features:
    X[f+'_norm'] = norm(arac[f].fillna(arac[f].median()))

# A: Esit agirlik
arac['skor_A_esit'] = X.mean(axis=1).round(2)

# B: r-orantili agirlik (r_dict'ten)
total_r = sum(r_dict.values())
weights_B = {f+'_norm': r_dict[f]/total_r for f in features}
print('=== POLITIKA B AGIRLIKLARI (r-orantili) ===')
for k, v in weights_B.items():
    print(f'  {k}: {v:.3f}')
arac['skor_B_r_orant'] = sum(X[col]*w for col, w in weights_B.items()).round(2)

# C: Logistic regression (test_risk_binary'yi tahmin et)
from sklearn.linear_model import LogisticRegression
X_train = X[arac['test_n_ariza'] > 0]
y_train = arac.loc[arac['test_n_ariza'] > 0, 'test_risk_binary']
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)
coefs = lr.coef_[0]
print(f'\n=== POLITIKA C AGIRLIKLARI (Logistic Regression) ===')
for col, coef in zip(X.columns, coefs):
    print(f'  {col}: {coef:+.4f}')
# Logistic regression olasilik tahmini
arac['skor_C_logreg'] = lr.predict_proba(X)[:,1] * 100  # 0-100 olcege
arac['skor_C_logreg'] = arac['skor_C_logreg'].round(2)

print('\n=== 3 SKORUN DAGILIMI ===')
print(arac[['skor_A_esit','skor_B_r_orant','skor_C_logreg']].describe().round(2))


=== POLITIKA B AGIRLIKLARI (r-orantili) ===
  yas_norm: 0.160
  gecmis_ciddi_oran_norm: 0.116
  gecmis_ort_skor_norm: 0.151
  egim_maruziyet_norm: 0.099
  garaj_ort_skor_norm: 0.354
  verimsizlik_skoru_norm: 0.121

=== POLITIKA C AGIRLIKLARI (Logistic Regression) ===
  yas_norm: -0.0083
  gecmis_ciddi_oran_norm: +0.0083
  gecmis_ort_skor_norm: -0.0071
  egim_maruziyet_norm: +0.0125
  garaj_ort_skor_norm: +0.0206
  verimsizlik_skoru_norm: +0.0077

=== 3 SKORUN DAGILIMI ===
       skor_A_esit  skor_B_r_orant  skor_C_logreg
count      3508.00         3508.00        3508.00
mean         46.95           53.64          51.25
std          12.70           13.23          11.67
min           4.80            2.85          15.19
25%          40.17           47.84          44.39
50%          46.83           54.40          51.84
75%          54.52           60.85          61.24
max          87.23           90.82          79.28


---
## 6. Skor Dağilimi + Q1-Q5 Bant Analizi

3 politikadan hangisinin bantları monotonik ve guclu fark gosteriyor?


In [6]:
# BOLUM 6: Bant analizi - 3 politikayi karsilastir
politikalar = ['skor_A_esit','skor_B_r_orant','skor_C_logreg']
for p in politikalar:
    arac[p+'_bant'] = pd.qcut(arac[p], q=5, labels=['Q1','Q2','Q3','Q4','Q5'], duplicates='drop')

print('=== BANT × TEST_CIDDI_ORAN ===')
for p in politikalar:
    print(f'\n--- {p} ---')
    bant = arac.groupby(p+'_bant', observed=True).agg(
        n=('KAPINO','count'),
        test_ciddi_oran=('test_ciddi_oran','mean'),
        test_ort_skor=('test_ort_skor','mean'),
    ).round(3)
    print(bant.to_string())
    # ANOVA F
    gruplar = [g['test_ort_skor'].values for _, g in arac.groupby(p+'_bant', observed=True)]
    f_stat, p_f = stats.f_oneway(*gruplar)
    print(f'ANOVA F={f_stat:.2f}, p={p_f:.6e}')
    # Monotoniklik: Q1 < Q5 mi?
    q1_oran = bant.iloc[0]['test_ciddi_oran']
    q5_oran = bant.iloc[-1]['test_ciddi_oran']
    print(f'Q1→Q5 monotonik artis: {q1_oran:.3f} → {q5_oran:.3f}, lift={q5_oran/q1_oran:.2f}x')


=== BANT × TEST_CIDDI_ORAN ===

--- skor_A_esit ---
                    n  test_ciddi_oran  test_ort_skor
skor_A_esit_bant                                     
Q1                705            0.289          3.080
Q2                702            0.364          3.531
Q3                699            0.365          3.559
Q4                700            0.394          3.673
Q5                702            0.410          3.736
ANOVA F=39.81, p=1.175721e-32
Q1→Q5 monotonik artis: 0.289 → 0.410, lift=1.42x

--- skor_B_r_orant ---
                       n  test_ciddi_oran  test_ort_skor
skor_B_r_orant_bant                                     
Q1                   704            0.285          3.037
Q2                   699            0.374          3.516
Q3                   703            0.369          3.561
Q4                   700            0.391          3.702
Q5                   702            0.403          3.761
ANOVA F=49.33, p=1.956844e-40
Q1→Q5 monotonik artis: 0.285 → 0.403, 

---
## 7. ROC + AUC Karşılaştırması

Hangi politika test_risk_binary'yi en iyi tahmin ediyor?


In [7]:
# BOLUM 7: AUC karsilastirmasi
valid = arac['test_n_ariza'] > 0
y = arac.loc[valid, 'test_risk_binary']

print('=== AUC KARSILASTIRMASI ===')
print(f'{"Yontem":25s} {"AUC":>8s}')
print('-'*40)
auc_yas = roc_auc_score(y, arac.loc[valid, 'yas'])
print(f'{"Baseline: yas":25s} {auc_yas:>8.4f}')

# Bireysel feature'lar
for f in ['gecmis_ciddi_oran','garaj_ort_skor','verimsizlik_skoru','egim_maruziyet']:
    auc_f = roc_auc_score(y, arac.loc[valid, f])
    delta = auc_f - auc_yas
    print(f'{f:25s} {auc_f:>8.4f}  (vs yas: {delta:+.4f})')

print()
for p in politikalar:
    auc_p = roc_auc_score(y, arac.loc[valid, p])
    delta = auc_p - auc_yas
    print(f'{p:25s} {auc_p:>8.4f}  (vs yas: {delta:+.4f})')

# En iyi politikayi sec
auc_sonuc = {p: roc_auc_score(y, arac.loc[valid, p]) for p in politikalar}
en_iyi = max(auc_sonuc, key=auc_sonuc.get)
print(f'\n=== KARAR ===')
print(f'En iyi politika: {en_iyi} (AUC={auc_sonuc[en_iyi]:.4f})')
arac['kritiklik_skoru'] = arac[en_iyi]
print(f'\nkritiklik_skoru = {en_iyi}')


=== AUC KARSILASTIRMASI ===
Yontem                         AUC
----------------------------------------
Baseline: yas               0.5288
gecmis_ciddi_oran           0.5653  (vs yas: +0.0365)
garaj_ort_skor              0.5607  (vs yas: +0.0319)
verimsizlik_skoru           0.6003  (vs yas: +0.0715)
egim_maruziyet              0.5769  (vs yas: +0.0481)

skor_A_esit                 0.5815  (vs yas: +0.0527)
skor_B_r_orant              0.5777  (vs yas: +0.0489)
skor_C_logreg               0.6235  (vs yas: +0.0947)

=== KARAR ===
En iyi politika: skor_C_logreg (AUC=0.6235)

kritiklik_skoru = skor_C_logreg


---
## 8. Random Null + Leakage Testi

Kompozit skor istatistiksel olarak rastgele degil mi? Zaman penceresinde stabil mi?


In [8]:
# BOLUM 8: Validation
# A) Random null - kritiklik_skoru × test_ort_skor
np.random.seed(42)
gercek_r, _ = stats.pearsonr(arac['kritiklik_skoru'], arac['test_ort_skor'])
print(f'Gercek r (kritiklik × test_ort_skor): {gercek_r:.4f}')

null = []
for _ in range(1000):
    perm = np.random.permutation(arac['test_ort_skor'].values)
    r, _ = stats.pearsonr(arac['kritiklik_skoru'], perm)
    null.append(r)
null = np.array(null)
print(f'Random null: mean={null.mean():.4f}, %95 CI=[{np.percentile(null,2.5):.4f}, {np.percentile(null,97.5):.4f}]')
print(f'Gercek > rastgele max: {gercek_r > null.max()}')

# B) Leakage testi: train donemindeki kritiklik_skoru -> test donemindeki ciddi_oran tahmini
# Train kritiklik skoru ile (zaten gecmis_ciddi_oran train'den geliyor) test'i tahmin et
print(f'\n=== STABILITE r ===')
# Train sub-period (ilk 1.5 ay) vs Test sub-period (son 1.5 ay)
df_sort = df.sort_values('OLAYTARIHI')
tarih_min = df_sort['OLAYTARIHI'].min()
tarih_max = df_sort['OLAYTARIHI'].max()
ortanca = tarih_min + (tarih_max - tarih_min) / 2
ilk_yari = df_sort[df_sort['OLAYTARIHI'] < ortanca]
ikinci_yari = df_sort[df_sort['OLAYTARIHI'] >= ortanca]

skor1 = ilk_yari.groupby('KAPINO')['ciddiyet_skoru'].mean()
skor2 = ikinci_yari.groupby('KAPINO')['ciddiyet_skoru'].mean()
ortak = skor1.index.intersection(skor2.index)
r_stab, _ = stats.pearsonr(skor1[ortak], skor2[ortak])
print(f'Arac ort_skor stability r (1.yari × 2.yari): {r_stab:.4f}')

# C) Time-aware AUC
auc_full = roc_auc_score(arac.loc[valid,'test_risk_binary'], arac.loc[valid,'kritiklik_skoru'])
print(f'\nKritiklik skor AUC (test_risk tahmini): {auc_full:.4f}')

# Dusus % - full vs time-aware
print(f'\n=== LEAKAGE DEGERLENDIRMESI ===')
if r_stab > 0.7:
    print(f'Stability r={r_stab:.2f} → SAGLAM (zaman icinde tutarli)')
elif r_stab > 0.5:
    print(f'Stability r={r_stab:.2f} → ORTA')
else:
    print(f'Stability r={r_stab:.2f} → ZAYIF (zaman icinde dalgalanma)')


Gercek r (kritiklik × test_ort_skor): 0.3036
Random null: mean=0.0006, %95 CI=[-0.0326, 0.0357]
Gercek > rastgele max: True

=== STABILITE r ===
Arac ort_skor stability r (1.yari × 2.yari): 0.1495

Kritiklik skor AUC (test_risk tahmini): 0.6235

=== LEAKAGE DEGERLENDIRMESI ===
Stability r=0.15 → ZAYIF (zaman icinde dalgalanma)


---
## 9. KISIM C: Top 50 / 100 / Tum Liste

Operasyonel mudahale önceligi.


In [9]:
# BOLUM 9: Top liste
arac_sirali = arac.sort_values('kritiklik_skoru', ascending=False).reset_index(drop=True)

print('=== TOP 50 ===')
top50 = arac_sirali.head(50)
print(top50[['KAPINO','GARAJ','MARKA','MODEL','yas','ort_skor','gecmis_ciddi_oran','kritiklik_skoru']].to_string(index=False))
print(f'\nTop 50 ozeti:')
print(f'  Ortalama yas: {top50["yas"].mean():.1f}')
print(f'  Ortalama ciddi_oran: {top50["ciddi_oran"].mean():.3f}')
print(f'  Filo genel ciddi_oran: {arac["ciddi_oran"].mean():.3f}')
print(f'  Lift: {top50["ciddi_oran"].mean()/arac["ciddi_oran"].mean():.2f}x')

print(f'\n=== TOP 100 OZETI ===')
top100 = arac_sirali.head(100)
print(f'  Ortalama yas: {top100["yas"].mean():.1f}')
print(f'  Ortalama ciddi_oran: {top100["ciddi_oran"].mean():.3f}')

# Tam liste CSV export
arac_sirali[['KAPINO','GARAJ','MARKA','MODEL','yas','ort_skor','ciddi_oran','gecmis_ciddi_oran','kritiklik_skoru']].to_csv('analiz8_tam_liste.csv', index=False)
print(f'\nTam liste CSV export edildi: analiz8_tam_liste.csv ({len(arac_sirali)} arac)')


=== TOP 50 ===
KAPINO      GARAJ    MARKA           MODEL  yas  ort_skor  gecmis_ciddi_oran  kritiklik_skoru
 M4579  Şahinkaya MERCEDES   CITARO 0530 G 19.0  4.110000           1.000000            79.28
 M6509  Şahinkaya MERCEDES   CITARO 0530 G 19.0  5.494000           1.000000            76.94
 M3100  Şahinkaya MERCEDES   CITARO 0530 G 19.0  5.006000           1.000000            76.42
 M4358  Şahinkaya MERCEDES   CITARO 0530 G 19.0  4.887000           1.000000            76.14
 M6512  Şahinkaya MERCEDES   CITARO 0530 G 19.0  4.892500           1.000000            76.07
 M3099  Şahinkaya MERCEDES   CITARO 0530 G 19.0  4.393636           1.000000            75.44
 K5791    KURTKÖY   KARSAN AVANCITY S PLUS 12.0  3.165556           1.000000            74.98
 M6314  Şahinkaya MERCEDES   CITARO 0530 G 19.0  4.710000           0.750000            74.29
 M2984  Şahinkaya MERCEDES   CITARO 0530 G 19.0  2.130000           0.500000            73.92
 M3202  Şahinkaya MERCEDES   CITARO 0530 G 19

---
## 10. Garaj × Kritiklik Dağılımı

Risk hangi garajlarda yogunlasiyor?


In [10]:
# BOLUM 10: Garaj x kritiklik
garaj_dag = arac.groupby('GARAJ').agg(
    n_arac=('KAPINO','count'),
    yas_ort=('yas','mean'),
    kritiklik_ort=('kritiklik_skoru','mean'),
    top50_sayi=('KAPINO', lambda s: s.isin(top50['KAPINO']).sum()),
).round(2).sort_values('kritiklik_ort', ascending=False)
print('=== GARAJ × KRITIKLIK ===')
print(garaj_dag.to_string())

# Top 50 dagilimi
print(f'\n=== TOP 50 GARAJ DAGILIMI ===')
print(top50['GARAJ'].value_counts())


=== GARAJ × KRITIKLIK ===
                           n_arac  yas_ort  kritiklik_ort  top50_sayi
GARAJ                                                                
Hasanpaşa                     319     8.03          64.44           3
Şahinkaya                     129    19.00          64.16          29
Edirnekapı                    381    11.83          63.69           8
KURTKÖY                       346    12.07          57.65          10
Kağıthane                     244    12.00          53.14           0
IKITELLIISLETTIRMEGARAJI2     330    11.32          51.35           0
IKITELLIGARAJI                429     9.23          49.67           0
SULTANGAZIGARAJI              413    12.24          47.79           0
Sarıgazi                      184    11.78          47.21           0
Anadolu                       353    17.36          43.01           0
Yunus                         229    12.00          39.42           0
Topkapı                       151     1.00          19.02       

---
## 11. Hat × Kritiklik

Yuksek riskli araclar hangi hatlarda calistiriliyor?


In [11]:
# BOLUM 11: Hat × kritiklik
# Top 50 araclarinin son 3 ay HATKODU dagilimi
top50_kapinos = top50['KAPINO'].tolist()
top50_hats = df[df['KAPINO'].isin(top50_kapinos)].groupby('HATKODU').size().reset_index(name='n_ariza').sort_values('n_ariza', ascending=False)
print('=== TOP 50 ARAC × HAT (en sik 15) ===')
print(top50_hats.head(15).to_string(index=False))

# Genel hat risk: hat × ortalama kritiklik
ah_recent = pd.read_csv('../panel_data/temiz_veri/arac_gunluk_hatlar.csv', low_memory=False)
arac_kritik = arac.set_index('KAPINO')['kritiklik_skoru'].to_dict()
ah_recent['kritiklik'] = ah_recent['KAPINO'].map(arac_kritik)
hat_risk = ah_recent.dropna(subset=['kritiklik']).groupby('HATKODU').agg(
    n_arac_kayit=('KAPINO','nunique'),
    kritiklik_ort=('kritiklik','mean'),
).round(2).sort_values('kritiklik_ort', ascending=False)
print(f'\n=== HAT × ORTALAMA KRITIKLIK (en kritik 15 hat) ===')
print(hat_risk.head(15).to_string())


=== TOP 50 ARAC × HAT (en sik 15) ===
HATKODU  n_ariza
    15F       83
   34BZ       59
   34AS       45
   11ÇB       41
    34G       38
   15BK       33
    18Ü       23
    18K       23
    11H       21
    34C       20
    18M       14
    16S       13
    19S       11
   121A        8
   KM24        8

=== HAT × ORTALAMA KRITIKLIK (en kritik 15 hat) ===
         n_arac_kayit  kritiklik_ort
HATKODU                             
121BS              76          69.39
15F               154          68.96
121A               91          68.56
15TA              110          67.61
15BK              181          65.73
500L                1          64.46
34A               345          64.27
34Z               698          64.02
34                672          64.01
34G               702          64.00
34BZ              705          63.98
34AS              700          63.98
34C               702          63.97
34B               654          63.96
SM9               149          63.37


---
## 12. Aksiyon Önerileri

Top 50 araclar icin kategorik aksiyon (yenileme / bakim / rotasyon).


In [12]:
# BOLUM 12: Aksiyon kategorileri
def aksiyon_belirle(row):
    yas = row['yas']
    skor = row['kritiklik_skoru']
    if yas >= 15:
        return 'MOTOR DURUMU DENETIMI (yasli + riskli, yenileme adayi)'
    elif yas >= 10 and row['gecmis_ciddi_oran'] > 0.5:
        return 'YOGUN BAKIM (orta yasli + yuksek ariza)'
    elif row['garaj_ort_skor'] > arac['garaj_ort_skor'].median():
        return 'GARAJ ROTASYONU (kotu garaj etkisi)'
    else:
        return 'IZLEME (orta risk)'

top50_kopya = top50.copy()
top50_kopya['aksiyon'] = top50_kopya.apply(aksiyon_belirle, axis=1)
print('=== TOP 50 AKSIYON DAGILIMI ===')
print(top50_kopya['aksiyon'].value_counts())
print()
print('=== TOP 50 ARAC × AKSIYON (ozet) ===')
print(top50_kopya[['KAPINO','GARAJ','yas','kritiklik_skoru','aksiyon']].to_string(index=False))


=== TOP 50 AKSIYON DAGILIMI ===
aksiyon
MOTOR DURUMU DENETIMI (yasli + riskli, yenileme adayi)    33
YOGUN BAKIM (orta yasli + yuksek ariza)                   10
GARAJ ROTASYONU (kotu garaj etkisi)                        7
Name: count, dtype: int64

=== TOP 50 ARAC × AKSIYON (ozet) ===
KAPINO      GARAJ  yas  kritiklik_skoru                                                aksiyon
 M4579  Şahinkaya 19.0            79.28 MOTOR DURUMU DENETIMI (yasli + riskli, yenileme adayi)
 M6509  Şahinkaya 19.0            76.94 MOTOR DURUMU DENETIMI (yasli + riskli, yenileme adayi)
 M3100  Şahinkaya 19.0            76.42 MOTOR DURUMU DENETIMI (yasli + riskli, yenileme adayi)
 M4358  Şahinkaya 19.0            76.14 MOTOR DURUMU DENETIMI (yasli + riskli, yenileme adayi)
 M6512  Şahinkaya 19.0            76.07 MOTOR DURUMU DENETIMI (yasli + riskli, yenileme adayi)
 M3099  Şahinkaya 19.0            75.44 MOTOR DURUMU DENETIMI (yasli + riskli, yenileme adayi)
 K5791    KURTKÖY 12.0            74.98         

---
## 13. ML V6 Feature: kritiklik_skoru Degerlendirmesi

Bireysel feature'larin uzerine ne kadar deger katiyor?


In [13]:
# BOLUM 13: ML feature degerlendirme
print('=== KRITIKLIK_SKORU ML FEATURE ===\n')
r_kritik, p_kritik = stats.pearsonr(arac['kritiklik_skoru'], arac['ort_skor'])
print(f'kritiklik_skoru × ort_skor: r={r_kritik:+.4f}, p={p_kritik:.6e}')

# Bireysel feature'larla karsilastirma
print(f'\nBireysel feature r degerleri:')
for f in ['yas','gecmis_ciddi_oran','garaj_ort_skor','verimsizlik_skoru','egim_maruziyet']:
    r_f, _ = stats.pearsonr(arac[f].fillna(0), arac['ort_skor'])
    print(f'  {f}: {r_f:+.4f}')
print(f'  KRITIKLIK_SKORU (kompozit): {r_kritik:+.4f}')

# AUC
auc_kritik = roc_auc_score(arac.loc[valid,'test_risk_binary'], arac.loc[valid,'kritiklik_skoru'])
print(f'\nAUC karsilastirmasi:')
print(f'  Sadece yas: {auc_yas:.4f}')
for f in ['gecmis_ciddi_oran','garaj_ort_skor','verimsizlik_skoru']:
    auc_f = roc_auc_score(arac.loc[valid,'test_risk_binary'], arac.loc[valid, f])
    print(f'  Sadece {f}: {auc_f:.4f}')
print(f'  KRITIKLIK_SKORU: {auc_kritik:.4f}')

# ANOVA bant
arac['kritiklik_bant'] = pd.qcut(arac['kritiklik_skoru'], q=5, labels=False, duplicates='drop')
gruplar = [g['ort_skor'].values for _, g in arac.groupby('kritiklik_bant')]
f_stat, p_f = stats.f_oneway(*gruplar)
print(f'\nANOVA bant (kritiklik 5-band): F={f_stat:.2f}, p={p_f:.6e}')


=== KRITIKLIK_SKORU ML FEATURE ===

kritiklik_skoru × ort_skor: r=+0.4794, p=4.785673e-201

Bireysel feature r degerleri:
  yas: +0.2092
  gecmis_ciddi_oran: +0.5899
  garaj_ort_skor: +0.4879
  verimsizlik_skoru: +0.1464
  egim_maruziyet: +0.1661
  KRITIKLIK_SKORU (kompozit): +0.4794

AUC karsilastirmasi:
  Sadece yas: 0.5288
  Sadece gecmis_ciddi_oran: 0.5653
  Sadece garaj_ort_skor: 0.5607
  Sadece verimsizlik_skoru: 0.6003
  KRITIKLIK_SKORU: 0.6235

ANOVA bant (kritiklik 5-band): F=167.77, p=1.340646e-131


---
## 14. Kısıtlamalar ve Sonraki Adımlar


In [14]:
# BOLUM 14: Kisitlamalar
print('=== KISITLAMALAR ===\n')
kisitlamalar = [
    '1. Veri penceresi 6 ay (Train 3 ay + Test 3 ay)',
    '   - Daha uzun donemde feature stabilitesi daha iyi test edilebilir',
    '',
    '2. yolcu_doluluk verisi güvenilmez (proje karari) -> kapasite feature yok',
    '',
    '3. ARACTIPI ayrımı yapılmadı (metrobus + otobus karisik)',
    '',
    '4. Onceki analizlerden tureyen feature\'lar burada bilinen confounder\'larla geliyor:',
    '   - egim_maruziyet (Analiz 3 metodu)',
    '   - garaj_ort_skor (Analiz 5 metodu, basitlestirildi)',
    '   - verimsizlik_skoru (Analiz 7 metodu)',
    '',
    '5. Logistic regression agirliklarinda overfitting riski',
    '   - Train/test split kullanildi ama daha buyuk veriyle CV onerilir',
    '',
    '6. ADALAR garajı bilerek dislandi (proje karari)',
    '',
    '7. Test_ciddi_oran tahmin hedefi - operasyonel risk taniminin proxy\'si',
    '   - Gercek risk: yolcu yarali kaza, plansiz cekme; bu veride ayni anlama gelen ciddi_ariza',
    '',
    '8. KRITIK: MODELYILI = uretim yili (motor durumunu YANSITMIYOR)',
    '   - IETT yasli araclara duzenli motor yenileme/bakim yapiyor',
    '   - "Yasli arac = riskli" varsayimi YANLIS olabilir',
    '   - yas feature AUC=0.529 (zayif) bunu destekliyor',
    '   - gecmis_ciddi_oran (r=+0.59) motor durumunu DOLAYLI olarak yakaliyor',
    '   - Aksiyon onerilerinde "yasli" yerine "performans bazli risk" daha guvenli',
]
for k in kisitlamalar:
    print(k)

print('\n=== SONRAKI ADIMLAR ===')
print('1. SONUCLAR.md yazimi')
print('2. Analiz 9 (Akilli Hat-Arac Atama) - synthesis adimi')
print('3. FEATURES_FINAL.md - tum kanitlanmis feature\'lari topla')
print('4. ML_MODEL_V6 - tum feature\'lari modele ekle, AUC karsilastir')


=== KISITLAMALAR ===

1. Veri penceresi 6 ay (Train 3 ay + Test 3 ay)
   - Daha uzun donemde feature stabilitesi daha iyi test edilebilir

2. yolcu_doluluk verisi güvenilmez (proje karari) -> kapasite feature yok

3. ARACTIPI ayrımı yapılmadı (metrobus + otobus karisik)

4. Onceki analizlerden tureyen feature'lar burada bilinen confounder'larla geliyor:
   - egim_maruziyet (Analiz 3 metodu)
   - garaj_ort_skor (Analiz 5 metodu, basitlestirildi)
   - verimsizlik_skoru (Analiz 7 metodu)

5. Logistic regression agirliklarinda overfitting riski
   - Train/test split kullanildi ama daha buyuk veriyle CV onerilir

6. ADALAR garajı bilerek dislandi (proje karari)

7. Test_ciddi_oran tahmin hedefi - operasyonel risk taniminin proxy'si
   - Gercek risk: yolcu yarali kaza, plansiz cekme; bu veride ayni anlama gelen ciddi_ariza

8. KRITIK: MODELYILI = uretim yili (motor durumunu YANSITMIYOR)
   - IETT yasli araclara duzenli motor yenileme/bakim yapiyor
   - "Yasli arac = riskli" varsayimi YANLIS 